# LunarLander with PyAOgmaNeo - Manual Implementation

VERSION NOTE: Master (commit: 379be3b)

## Introduction

This example trains a **PyAOgmaNeo** agent to solve the LunarLander control problem, showing how SPH (Sparse Predictive Hierarchies) handles a more complex environment than CartPole.

**The Task:** Land a spacecraft safely on the moon's surface between two flags by controlling thrusters.

**What you'll learn:**
1. Converting 8 continuous observations to CSDR format
2. Setting up a hierarchy for a complex task
3. Training with environmental reward shaping
4. Evaluating performance

> **New to neuromorphic computing?** See [History of Neuromorphic Computing](../../overview/history_of_neuromorphic_computing.md) for background.


## 0. Dependencies

**Required:** PyAOgmaNeo, Gymnasium (with Box2D), NumPy, Matplotlib

**⚠️ IMPORTANT - Install SWIG first:**

Box2D requires SWIG to be installed at the system level. **Run this in your terminal** (not in the notebook):

```bash
sudo apt-get update && sudo apt-get install -y swig
```

After SWIG is installed, run the next cell to install Python packages.

> See [Installation Guide](../../getting_started/installation.md) for troubleshooting.


In [12]:
# Install Python dependencies
# Make sure SWIG is installed first (see cell above)
%pip install 'gymnasium[box2d]' numpy matplotlib pyaogmaneo


  Using cached box2d-py-2.3.5.tar.gz (374 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pygame-2.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached swig-4.4.0-py3-none-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (3.5 kB)
Using cached swig-4.4.0-py3-none-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (1.9 MB)
Using cached pygame-2.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (14.0 MB)
  Created wheel for box2d-py: filename=box2d_py-2.3.5-cp311-cp311-linux_x86_64.whl size=2807199 sha256=7b127755a2eb0e21be94ce8c1c7be600d5eebdbb8ac067abc6619916fa4e12b3
  Stored in directory: /home/izack/.cache/pip/wheels/ab/f1/0c/d56f4a2bdd12bae0a0693ec33f2f0daadb5eb9753c78fa5308
Successfully built box2d-py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pygame]2m2/3 [pygame]y]

[notice] A new release of pip is available: 25.1

## 1. Setup and Imports

Import the core libraries:
- **pyaogmaneo** - The SPH neural network
- **gymnasium** - RL environment interface
- **numpy** - Array operations and encoding
- **matplotlib** - Visualization


In [13]:
import pyaogmaneo as neo
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Configuration

### CSDR Encoding

- **input_resolution = 32:** Discretization bins per observation
- **encoding_scale = 3.0:** Sensitivity multiplier for sigmoid encoding

### Architecture

- **input_size = (4, 2, 32):** 8 observations in 4×2 grid, 32 values each
- **hidden_size = (6, 6, 32):** 6×6 spatial structure per layer
- **num_layers = 2:** Two hierarchy layers (immediate + temporal patterns)

### Training

- **max_episodes = 5000:** Sufficient for LunarLander convergence
- **max_timesteps = 10000:** Max steps per episode
- **reward_scale = 1.0:** Use environment rewards as-is

> **Learn more:** [CSDR concepts](../../technical_guide/core_concepts.md#csdr-the-common-language) | [Parameter Tuning Guide](../../technical_guide/parameter_tuning.md)


In [14]:
# 2. Configuration

# CSDR encoding parameters
input_resolution = 32      # Number of discrete bins per observation
encoding_scale = 3.0       # Sensitivity multiplier for sigmoid

# SPH architecture
input_size = (4, 2, input_resolution)   # 8 observations in 4×2 grid
hidden_size = (6, 6, 32)                # Hidden layer size
num_layers = 2                           # Number of hierarchy layers

# Training schedule
max_episodes = 5000        # Number of training episodes
max_timesteps = 10000      # Max steps per episode
reward_scale = 1.0         # Reward multiplier

# Parallelism for PyAOgmaNeo
neo.set_num_threads(8)

print(f"Configured for {max_episodes} training episodes.")
print(f"Input: {input_size}, Hidden: {hidden_size}, Layers: {num_layers}")


Configured for 5000 training episodes.
Input: (4, 2, 32), Hidden: (6, 6, 32), Layers: 2


## 3. LunarLander Environment

**Task:** Land a spacecraft softly on the pad between two flags.

### Observations (8 continuous values)

| Index | Observation | Description |
|-------|-------------|-------------|
| 0-1 | Position (x, y) | Horizontal and vertical position |
| 2-3 | Velocity (x, y) | Horizontal and vertical velocity |
| 4-5 | Angle, Angular Velocity | Orientation and rotation rate |
| 6-7 | Leg Contacts | Left and right leg touching ground |

### Actions (4 discrete)

- `0`: Do nothing
- `1`: Fire left engine
- `2`: Fire main engine (up)
- `3`: Fire right engine

### Rewards

- Landing on pad: +100 to +140
- Crashing: -100
- Fuel usage: Small penalties
- **Solved:** Average reward ≥ 200 over 100 episodes

> [Gymnasium LunarLander docs](https://gymnasium.farama.org/environments/box2d/lunar_lander/)


In [15]:
# 3. Create the LunarLander environment

env = gym.make('LunarLander-v3')  # Use render_mode='human' to visualize

num_obs = env.observation_space.shape[0]  # Should be 8
num_actions = env.action_space.n           # Should be 4

print(f"Environment: LunarLander-v3")
print(f"Observation space: {num_obs} continuous values")
print(f"  - Position (x, y)")
print(f"  - Velocity (x, y)")
print(f"  - Angle, angular velocity")
print(f"  - Leg contacts (left, right)")
print(f"Action space: {num_actions} discrete actions")
print(f"  - 0: Nothing, 1: Left, 2: Main, 3: Right")
print(f"Goal: Land safely on the pad for +200 average reward")


Environment: LunarLander-v3
Observation space: 8 continuous values
  - Position (x, y)
  - Velocity (x, y)
  - Angle, angular velocity
  - Leg contacts (left, right)
Action space: 4 discrete actions
  - 0: Nothing, 1: Left, 2: Main, 3: Right
Goal: Land safely on the pad for +200 average reward


## 4. CSDR Encoding - The "Squash and Bin" Method

Convert continuous observations to discrete CSDR format for PyAOgmaNeo.

### The Method

**Step 1: Squash** - Map any value to [0, 1] using sigmoid
**Step 2: Bin** - Discretize to integer bins (0-31)

```python
csdr = (sigmoid(obs * encoding_scale) * (input_resolution - 1) + 0.5).astype(np.int32)
```

This encoding:
- Handles unbounded inputs (velocities can be large)
- Preserves ordering (larger values → larger bins)
- Produces consistent output range regardless of input magnitude

> **Deep dive:** [CSDR: The Common Language](../../technical_guide/core_concepts.md#csdr-the-common-language)


In [16]:
# 4. Define the "squash and bin" encoding function

def sigmoid(x):
    """Squash any input to [0, 1] range using smooth S-curve."""
    return np.tanh(x * 0.5) * 0.5 + 0.5

def encode_observation(obs):
    """Convert continuous observations to discrete CSDR format.
    
    Args:
        obs: Array of 8 continuous values
        
    Returns:
        Array of 8 discrete integers (0 to input_resolution-1)
    """
    csdr = (sigmoid(obs * encoding_scale) * (input_resolution - 1) + 0.5).astype(np.int32)
    return csdr

# Test the encoding with a sample observation
test_obs = np.array([0.2, 1.5, -0.3, 0.8, 0.0, -1.2, 0.0, 1.0])
test_encoded = encode_observation(test_obs)

print("Encoding demonstration:")
print(f"Raw observations:     {test_obs}")
print(f"After sigmoid:        {sigmoid(test_obs * encoding_scale)}")
print(f"Encoded to bins:      {test_encoded}")
print(f"\nEach value is now a discrete bin (0-{input_resolution-1})")


Encoding demonstration:
Raw observations:     [ 0.2  1.5 -0.3  0.8  0.  -1.2  0.   1. ]
After sigmoid:        [0.64565631 0.98901306 0.2890505  0.9168273  0.5        0.02659699
 0.5        0.95257413]
Encoded to bins:      [20 31  9 28 16  1 16 30]

Each value is now a discrete bin (0-31)


## 5. Hierarchy Construction

Build the SPH agent using the Hierarchy API.

### I/O Streams

- **Stream 0:** Observations (4×2×32 CSDR grid, type `neo.none`)
- **Stream 1:** Actions (1×1×4 discrete, type `neo.action` for RL)

### Layers

- **Hidden size:** (6, 6, 32) per layer
- **Layer 0:** Learns immediate state transitions
- **Layer 1:** Learns longer temporal patterns

> [Hierarchy API Reference](../../api_reference/hierarchy.md)


In [17]:
# 5. Build the SPH agent architecture

# Define layer descriptors
lds = []
for i in range(num_layers):
    ld = neo.LayerDesc()
    ld.hidden_size = hidden_size
    lds.append(ld)

# Create hierarchy with two IO streams:
#   - Stream 0: Observations (4×2×32 CSDR)
#   - Stream 1: Actions (1×1×4 discrete)
h = neo.Hierarchy(
    [
        neo.IODesc(input_size, neo.none),                  # Observations
        neo.IODesc((1, 1, num_actions), neo.action),       # Actions
    ],
    lds,
)

print("Hierarchy created successfully!")
print(f"  Input size:  {input_size}")
for i in range(num_layers):
    print(f"  Layer {i} hidden size: {h.get_hidden_size(i)}")
print(f"  Action space: {num_actions} discrete actions")
print(f"\nThe agent is ready to learn!")


Hierarchy created successfully!
  Input size:  (4, 2, 32)
  Layer 0 hidden size: (6, 6, 32)
  Layer 1 hidden size: (6, 6, 32)
  Action space: 4 discrete actions

The agent is ready to learn!


## 6. Training Loop - Online Predictive Learning

### The Learning Cycle

```python
csdr = encode_observation(obs)                              # 1. Encode
h.step([csdr, h.get_prediction_cis(1)], True, reward)      # 2. Learn & predict
action = h.get_prediction_cis(1)[0]                         # 3. Get action
obs, reward, term, trunc, _ = env.step(action)              # 4. Execute
```

The hierarchy:
- Compares predictions with actual observations
- Updates based on prediction error and reward
- Learns fully online (no replay buffer, no batches)

LunarLander provides built-in reward shaping (landing bonus, crash penalty, fuel costs), so we use rewards as-is.

> [Reinforcement Learning Guide](../../user_guide/reinforcement_learning.md)


In [18]:
# 6. Training loop implementation

def train_agent():
    """Train the LunarLander agent and return training statistics."""
    
    episode_lengths = []
    episode_rewards = []
    
    print(f"Starting training for {max_episodes} episodes...\n")
    
    reward = 0.0  # Initial reward for first timestep
    
    for episode in range(max_episodes):
        obs, _ = env.reset()
        total_reward = 0.0
        
        for t in range(max_timesteps):
            # 1. Encode observations to CSDR
            csdr = encode_observation(obs)
            
            # 2. Step hierarchy: learn from previous reward, generate predictions
            h.step([csdr, h.get_prediction_cis(1)], True, reward * reward_scale)
            
            # 3. Get predicted action (exploration already applied)
            action = h.get_prediction_cis(1)[0]
            
            # 4. Execute action in environment
            obs, reward, term, trunc, _ = env.step(action)
            
            total_reward += reward
            
            if term or trunc:
                break
        
        # Record statistics
        episode_lengths.append(t + 1)
        episode_rewards.append(total_reward)
        
        # Print progress every 100 episodes
        if (episode + 1) % 100 == 0:
            avg_length = np.mean(episode_lengths[-100:])
            avg_reward = np.mean(episode_rewards[-100:])
            max_reward = np.max(episode_rewards[-100:])
            print(f"Episode {episode + 1:4d}: avg length = {avg_length:6.1f}, avg reward = {avg_reward:7.2f}, max reward = {max_reward:7.2f}")
    
    print("\nTraining complete!")
    
    return np.array(episode_lengths), np.array(episode_rewards)

# Run training
episode_lengths, episode_rewards = train_agent()


Starting training for 5000 episodes...

Episode  100: avg length =   81.7, avg reward = -156.27, max reward =   16.26
Episode  200: avg length =  100.8, avg reward = -118.02, max reward =   27.96
Episode  300: avg length =  257.3, avg reward = -113.74, max reward =  268.82
Episode  400: avg length =  411.9, avg reward =  -59.87, max reward =  247.23
Episode  500: avg length =  298.0, avg reward =  -18.68, max reward =  274.89
Episode  600: avg length =  233.0, avg reward =  -41.18, max reward =  260.15
Episode  700: avg length =  290.5, avg reward =  -76.55, max reward =  248.79
Episode  800: avg length =  277.0, avg reward =  -34.21, max reward =  250.12


KeyboardInterrupt: 

## 7. Visualization - Training Performance

Plot episode lengths and rewards to track learning progress.

**Success criteria:** Average reward ≥ 200 over 100 episodes


In [ ]:
# 7. Plot training curves

episodes = np.arange(1, len(episode_lengths) + 1)

# Calculate moving average (window=100)
window = 100
moving_avg = np.convolve(episode_lengths, np.ones(window)/window, mode='valid')

# Plot episode lengths
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(episodes, episode_lengths, alpha=0.3, label='Episode length')
plt.plot(episodes[window-1:], moving_avg, linewidth=2, label=f'Moving avg ({window} episodes)')
plt.xlabel('Episode')
plt.ylabel('Episode Length (timesteps)')
plt.title('LunarLander Training Progress - Episode Length')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot rewards
plt.subplot(1, 2, 2)
plt.plot(episodes, episode_rewards, alpha=0.3, label='Episode reward')
moving_avg_reward = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
plt.plot(episodes[window-1:], moving_avg_reward, linewidth=2, label=f'Moving avg ({window} episodes)')
plt.axhline(y=200, color='r', linestyle='--', label='Solved threshold (200)')
plt.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('LunarLander Training Progress - Reward')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final statistics
final_avg_length = np.mean(episode_lengths[-100:])
final_avg_reward = np.mean(episode_rewards[-100:])
final_max_reward = np.max(episode_rewards[-100:])
print(f"\nFinal 100 episodes:")
print(f"  Average length: {final_avg_length:.1f} timesteps")
print(f"  Average reward: {final_avg_reward:.2f}")
print(f"  Max reward:     {final_max_reward:.2f}")
if final_avg_reward >= 200:
    print(f"  Task SOLVED! (threshold: 200)")
else:
    print(f"  Not yet solved (threshold: 200)")


## 8. Testing the Trained Agent

Evaluate performance with learning disabled (`learn=False`) to see the agent's true capability without exploration noise.


In [ ]:
# 8. Test the trained agent

def test_agent(num_test_episodes=10):
    """Evaluate the trained agent without learning."""
    
    test_lengths = []
    test_rewards = []
    
    print(f"\nTesting agent for {num_test_episodes} episodes (no learning)...\n")
    
    for episode in range(num_test_episodes):
        obs, _ = env.reset()
        total_reward = 0.0
        
        for t in range(max_timesteps):
            csdr = encode_observation(obs)
            h.step([csdr, h.get_prediction_cis(1)], False, 0.0)
            action = h.get_prediction_cis(1)[0]
            obs, reward, term, trunc, _ = env.step(action)
            total_reward += reward
            
            if term or trunc:
                break
        
        test_lengths.append(t + 1)
        test_rewards.append(total_reward)
        print(f"Test episode {episode + 1}: {t + 1} timesteps, reward = {total_reward:.2f}")
    
    avg_length = np.mean(test_lengths)
    avg_reward = np.mean(test_rewards)
    
    print(f"\nTest results over {num_test_episodes} episodes:")
    print(f"  Average length: {avg_length:.1f} timesteps")
    print(f"  Average reward: {avg_reward:.2f}")
    print(f"  Best reward:    {max(test_rewards):.2f}")
    print(f"  Worst reward:   {min(test_rewards):.2f}")
    
    return test_lengths, test_rewards

# Run evaluation
test_lengths, test_rewards = test_agent(num_test_episodes=10)


## 9. Analysis and Next Steps

### What the Agent Learned

- Predict how thrusters affect position and velocity
- Coordinate multiple controls (main + orientation engines)
- Plan multi-step landing sequences
- Optimize fuel usage

### SPH Advantages

- Online learning (no replay buffer)
- CPU-only training (no GPU needed)
- Real-time adaptation capability

### Experiments to Try

1. Adjust `encoding_scale` or `input_resolution`
2. Try different `hidden_size` or add a third layer
3. Apply to LunarLanderContinuous

> **Simpler approach:** See [EnvRunner Implementation](../cartpole/env_runner_implementation.md) for automatic setup.


## 10. Cleanup and Model Persistence


In [ ]:
# 10. Cleanup

env.close()
print("Environment closed.")
print("\nTo save the trained model:")
print("  h.save_to_file('lunarlander_manual.ohr')")
print("\nTo load it later:")
print("  h = neo.Hierarchy(file_name='lunarlander_manual.ohr')")
